# 03 — Baseline & GCN Training

Train the heuristic baseline (common neighbors) and the 2-layer GCN on the drug-drug interaction graph. Evaluate on validation and test sets.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import torch
import numpy as np
import matplotlib.pyplot as plt

from src.preprocessing import load_processed
from src.graph_builder import build_homo_data
from src.models import GCNLinkPredictor, compute_common_neighbor_scores
from src.training import train_gcn, sample_negatives
from src.evaluation import compute_link_scores, compute_metrics, compute_curves, compute_ranking_metrics
from src.plotting import plot_training_curves, plot_roc_curves, plot_pr_curves
from src.utils import (
    set_seed, get_device, save_checkpoint, save_metrics,
    DATA_SPLITS, BLOG_ASSETS
)

%matplotlib inline

In [ ]:
SEED = 42
set_seed(SEED)
device = get_device()
print(f"Device: {device}")

# Load processed data
id_maps, edges, stats = load_processed()
homo_ei, num_drugs = build_homo_data(id_maps, edges)
print(f"Drugs: {num_drugs}, DDI edges: {homo_ei.shape[1]}")

# Load splits
train_edges = torch.load(os.path.join(DATA_SPLITS, 'train_edges.pt'), weights_only=True)
val_edges = torch.load(os.path.join(DATA_SPLITS, 'val_edges.pt'), weights_only=True)
test_edges = torch.load(os.path.join(DATA_SPLITS, 'test_edges.pt'), weights_only=True)
train_edges_ud = torch.load(os.path.join(DATA_SPLITS, 'train_edges_ud.pt'), weights_only=True)

print(f"Train: {train_edges.shape[1]}, Val: {val_edges.shape[1]}, Test: {test_edges.shape[1]}")

## 1. Heuristic Baseline (Common Neighbors)

In [ ]:
cn_scores = compute_common_neighbor_scores(train_edges_ud, num_drugs)

# Evaluate on test set
set_seed(SEED)
neg_test = sample_negatives(test_edges, num_drugs, test_edges.shape[1] * 5)

pos_cn = cn_scores[test_edges[0].numpy(), test_edges[1].numpy()]
neg_cn = cn_scores[neg_test[0].numpy(), neg_test[1].numpy()]

heuristic_metrics = compute_metrics(
    torch.tensor(pos_cn, dtype=torch.float),
    torch.tensor(neg_cn, dtype=torch.float)
)
print(f"Heuristic baseline (common neighbors):")
print(f"  AUROC: {heuristic_metrics['auroc']:.4f}")
print(f"  AUPRC: {heuristic_metrics['auprc']:.4f}")

heuristic_curves = compute_curves(
    torch.tensor(pos_cn, dtype=torch.float),
    torch.tensor(neg_cn, dtype=torch.float)
)

## 2. GCN Training

In [ ]:
set_seed(SEED)
gcn = GCNLinkPredictor(num_drugs, embed_dim=64, dropout=0.3).to(device)
optimizer = torch.optim.Adam(gcn.parameters(), lr=0.01)

gcn_history, gcn_best_auroc = train_gcn(
    gcn, optimizer,
    train_ei=train_edges,
    val_ei=val_edges,
    graph_ei=train_edges_ud,
    num_nodes=num_drugs,
    epochs=200,
    patience=20,
    neg_ratio=5,
    device=device,
)
print(f"\nBest validation AUROC: {gcn_best_auroc:.4f}")
save_checkpoint(gcn, f'gcn_base_s{SEED}')

## 3. GCN Evaluation

In [ ]:
gcn.eval()

# Score test edges
set_seed(SEED)
neg_test = sample_negatives(test_edges, num_drugs, test_edges.shape[1] * 5)

pos_scores = compute_link_scores(gcn, test_edges, model_type='gcn',
                                  graph_edge_index=train_edges_ud, device=device)
neg_scores = compute_link_scores(gcn, neg_test, model_type='gcn',
                                  graph_edge_index=train_edges_ud, device=device)

gcn_metrics = compute_metrics(pos_scores, neg_scores)
gcn_ranking = compute_ranking_metrics(gcn, test_edges, num_drugs, model_type='gcn',
                                      graph_edge_index=train_edges_ud, device=device)
gcn_metrics.update(gcn_ranking)
gcn_curves = compute_curves(pos_scores, neg_scores)

print("GCN Test Metrics:")
for k, v in gcn_metrics.items():
    print(f"  {k}: {v:.4f}")

save_metrics(f'gcn_base_s{SEED}', gcn_metrics)

## 4. Training Curves

In [ ]:
fig = plot_training_curves({'gcn': gcn_history}, metric='train_loss')
plt.show()

fig = plot_training_curves({'gcn': gcn_history}, metric='val_auroc')
plt.show()

## 5. Save Curves for Later Comparison

In [ ]:
import json
from src.utils import RESULTS_DIR

curves_data = {
    'heuristic': {k: v.tolist() if hasattr(v, 'tolist') else v for k, v in heuristic_curves.items()},
    'gcn': {k: v.tolist() if hasattr(v, 'tolist') else v for k, v in gcn_curves.items()},
}
with open(os.path.join(RESULTS_DIR, 'curves_data.json'), 'w') as f:
    json.dump(curves_data, f)

print("Curves saved for later overlay with RGCN.")